In [1]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import mlflow
from pathlib import Path
from collections import defaultdict
from datetime import datetime

from src.utils.config import settings
from src.utils.logger import get_logger

log = get_logger("multitask_retraining")

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu')

PROC = '../../data/processed/'
FEAT = '../../data/features/'

print(f"✅ Imports ready")
print(f"   Device  : {device}")
print(f"   PyTorch : {torch.__version__}")

✅ Imports ready
   Device  : cpu
   PyTorch : 2.3.0


In [2]:
#load data
ratings = pd.read_csv(
    PROC + 'ratings_cleaned.csv')
movies  = pd.read_csv(
    PROC + 'movies_master.csv',
    low_memory=False)
movies  = movies[
    movies['movieId'].notna()].copy()
movies['movieId'] = \
    movies['movieId'].astype(int)

vocab = joblib.load(
    '../../models/checkpoints/'
    'item_vocabulary.joblib')
user_sequences = joblib.load(
    '../../models/checkpoints/'
    'user_sequences.joblib')

PAD_TOKEN  = vocab['PAD']
OFFSET     = 3
vocab_size = vocab['vocab_size']
movie2token = vocab['movie2token']
token2movie = {
    int(k): v
    for k, v in vocab['token2movie'].items()
}

ratings = ratings.sort_values(
    ['userId', 'timestamp']
).reset_index(drop=True)

# Temporal split
n         = len(ratings)
train_end = int(n * 0.8)
val_end   = int(n * 0.9)

train_df = ratings.iloc[:train_end].copy()
val_df   = ratings.iloc[train_end:val_end].copy()
test_df  = ratings.iloc[val_end:].copy()

print(f"✅ Data loaded")
print(f"   Train : {len(train_df):,}")
print(f"   Val   : {len(val_df):,}")
print(f"   Test  : {len(test_df):,}")

✅ Data loaded
   Train : 80,003
   Val   : 10,000
   Test  : 10,001


In [3]:
#Simulate Watch Completion
def simulate_watch_completion(rating: float,
                               noise: float = 0.1
                               ) -> float:
    """
    Simulate watch completion percentage.

    In production Netflix has actual watch time.
    We simulate it from ratings:
    Rating 5.0 → ~95% completion
    Rating 4.0 → ~75% completion
    Rating 3.0 → ~50% completion
    Rating 2.0 → ~25% completion
    Rating 1.0 → ~10% completion

    Add gaussian noise to simulate real variance:
    Some users finish bad movies, some quit good ones
    """
    base = {
        5.0: 0.95, 4.5: 0.85,
        4.0: 0.75, 3.5: 0.60,
        3.0: 0.50, 2.5: 0.35,
        2.0: 0.25, 1.5: 0.15,
        1.0: 0.10, 0.5: 0.05,
    }
    # Find closest rating
    closest = min(base.keys(),
                  key=lambda x: abs(x - rating))
    completion = base[closest]

    # Add noise
    completion += np.random.normal(0, noise)
    return float(np.clip(completion, 0.0, 1.0))


# Add watch completion to ratings
np.random.seed(42)
ratings['watch_completion'] = ratings[
    'rating'].apply(simulate_watch_completion)

# Update splits
train_df = ratings.iloc[:train_end].copy()
val_df   = ratings.iloc[train_end:val_end].copy()
test_df  = ratings.iloc[val_end:].copy()

print(f"✅ Watch completion simulated")
print(f"\nSample data:")
print(ratings[['userId', 'movieId',
               'rating',
               'watch_completion']].head(8))
print(f"\nWatch completion stats:")
print(ratings['watch_completion'].describe(
).round(3))

✅ Watch completion simulated

Sample data:
   userId  movieId  rating  watch_completion
0       1     2294     2.0          0.299671
1       1     2455     2.5          0.336174
2       1     3671     3.0          0.564769
3       1     1339     3.5          0.752303
4       1     1343     2.0          0.226585
5       1     1371     2.5          0.326586
6       1     2105     4.0          0.907921
7       1       31     2.5          0.426743

Watch completion stats:
count    100004.000
mean          0.626
std           0.253
min           0.000
25%           0.461
50%           0.661
75%           0.824
max           1.000
Name: watch_completion, dtype: float64


In [4]:
#Multi-task Dataset
class MultiTaskDataset(Dataset):
    """
    Dataset for explicit multi-task ranking.

    Two prediction targets per interaction:
    1. Rating (0.5 – 5.0)
       → Did user like the movie?
    2. Watch completion (0.0 – 1.0)
       → Did user finish the movie?

    Why both matter:
    Rating alone  → recommends beloved classics
                    users never finish
    Watch% alone  → recommends short films or
                    films users felt guilty stopping
    Both together → recommends films users
                    genuinely enjoy AND finish
    """

    def __init__(self,
                 ratings_df:    pd.DataFrame,
                 user_sequences: dict,
                 movie2token:   dict,
                 max_len:       int = 50,
                 n_neg:         int = 4):
        self.samples    = []
        self.max_len    = max_len
        self.all_tokens = list(
            movie2token.values())

        # Build rating + completion lookup
        self.rating_lookup     = defaultdict(dict)
        self.completion_lookup = defaultdict(dict)

        for _, row in ratings_df.iterrows():
            mid = row['movieId']
            if mid in movie2token:
                tok = movie2token[mid]
                uid = row['userId']
                self.rating_lookup[uid][tok] = \
                    row['rating']
                self.completion_lookup[uid][tok]=\
                    row['watch_completion']

        # Build samples from sequences
        for uid, seq in user_sequences.items():
            if len(seq) < 3:
                continue

            for end_idx in range(2, len(seq)):
                history = seq[
                    max(0,
                        end_idx-max_len
                        ):end_idx]
                target  = seq[end_idx] \
                    if end_idx < len(seq) \
                    else seq[-1]

                rating = self.rating_lookup\
                    .get(uid, {})\
                    .get(target, 3.5)
                completion = self.completion_lookup\
                    .get(uid, {})\
                    .get(target, 0.5)

                self.samples.append({
                    'history':     history,
                    'target':      target,
                    'rating':      rating,
                    'completion':  completion,
                    'history_set': set(seq[:end_idx])
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        pad_len = self.max_len - len(s['history'])
        padded  = [PAD_TOKEN] * pad_len + \
                   s['history']

        # Sample negatives
        negs     = []
        attempts = 0
        while len(negs) < 4 and attempts < 200:
            t = np.random.choice(self.all_tokens)
            if t not in s['history_set']:
                negs.append(t)
            attempts += 1
        while len(negs) < 4:
            negs.append(
                np.random.choice(self.all_tokens))

        return {
            'history':    torch.LongTensor(padded),
            'target':     torch.LongTensor(
                              [s['target']]),
            'negatives':  torch.LongTensor(negs),
            'rating':     torch.FloatTensor(
                              [s['rating']]),
            'completion': torch.FloatTensor(
                              [s['completion']]),
        }


# Split users
all_users = list(user_sequences.keys())
np.random.seed(42)
np.random.shuffle(all_users)
split     = int(len(all_users) * 0.9)

train_users = {
    u: user_sequences[u]
    for u in all_users[:split]
}
val_users   = {
    u: user_sequences[u]
    for u in all_users[split:]
}

train_ds = MultiTaskDataset(
    train_df, train_users, movie2token)
val_ds   = MultiTaskDataset(
    val_df, val_users, movie2token)

train_dl = DataLoader(
    train_ds, batch_size=256,
    shuffle=True, num_workers=0)
val_dl   = DataLoader(
    val_ds, batch_size=256,
    shuffle=False, num_workers=0)

print(f"✅ Multi-task dataset ready")
print(f"   Train : {len(train_ds):,}")
print(f"   Val   : {len(val_ds):,}")

✅ Multi-task dataset ready
   Train : 20,875
   Val   : 2,330


In [6]:
#Explicit Multi-task Ranker
class ExplicitMultiTaskRanker(nn.Module):
    """
    Explicit Multi-task Ranker.

    This is what Netflix actually does in
    production — not just predict next item
    but predict MULTIPLE quality signals
    simultaneously.

    Architecture:
    1. Shared encoder (transformer)
       → learns general user preferences

    2. Task-specific towers
       → Rating tower: predicts star rating
       → Completion tower: predicts watch%
       → Ranking tower: BPR item scoring

    3. Combined output
       → weighted combination of all signals

    Why shared encoder + separate towers:
    → Shared: user preference patterns
              are common across tasks
    → Separate: rating ≠ completion signal
                need different output layers
    """

    def __init__(self,
                 vocab_size:   int,
                 embed_dim:    int   = 128,
                 n_heads:      int   = 4,
                 n_layers:     int   = 2,
                 max_seq_len:  int   = 50,
                 dropout:      float = 0.1):
        super().__init__()

        self.embed_dim = embed_dim

        # Shared item embeddings
        self.item_emb = nn.Embedding(
            vocab_size, embed_dim,
            padding_idx=PAD_TOKEN)
        self.pos_emb  = nn.Embedding(
            max_seq_len, embed_dim)

        # Shared transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = embed_dim,
            nhead           = n_heads,
            dim_feedforward = embed_dim * 4,
            dropout         = dropout,
            batch_first     = True,
            norm_first      = True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers)

        self.norm    = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

        # Task 1: Rating prediction tower
        # Predicts 0.5 – 5.0
        self.rating_tower = nn.Sequential(
            nn.Linear(embed_dim * 2,
                      embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim,
                      embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, 1),
            nn.Sigmoid(),
        )

        # Task 2: Watch completion tower
        # Predicts 0.0 – 1.0
        self.completion_tower = nn.Sequential(
            nn.Linear(embed_dim * 2,
                      embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim,
                      embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, 1),
            nn.Sigmoid(),
        )

        # Task 3: Ranking tower
        # BPR-style item scoring
        self.ranking_tower = nn.Sequential(
            nn.Linear(embed_dim * 2,
                      embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1),
        )

    def encode_user(self, history):
        B, L = history.shape
        positions = torch.arange(
            L, device=history.device
        ).unsqueeze(0).expand(B, -1)

        x = self.dropout(
            self.item_emb(history) +
            self.pos_emb(positions))

        pad_mask = (history == PAD_TOKEN)
        x = self.encoder(
            x,
            src_key_padding_mask=pad_mask)
        x = self.norm(x)

        lengths  = (history != PAD_TOKEN)\
            .sum(dim=1) - 1
        lengths  = lengths.clamp(min=0)
        return x[torch.arange(B), lengths]

    def predict_all_tasks(self,
                           user_repr,
                           item_emb):
        """
        Predict all three tasks for
        user-item pair.
        """
        combined = torch.cat(
            [user_repr, item_emb], dim=-1)

        rating     = self.rating_tower(
            combined).squeeze(-1)
        # Scale to 0.5-5.0
        rating     = rating * 4.5 + 0.5

        completion = self.completion_tower(
            combined).squeeze(-1)

        rank_score = self.ranking_tower(
            combined).squeeze(-1)

        return rating, completion, rank_score

    def forward(self, history,
                pos_items, neg_items):
        user_repr = self.encode_user(history)

        pos_emb   = self.item_emb(pos_items)
        neg_emb   = self.item_emb(neg_items)

        # All tasks for positive items
        pos_rating, pos_completion, \
        pos_rank = self.predict_all_tasks(
            user_repr, pos_emb)

        # Ranking score for negatives
        neg_combined = torch.cat(
            [user_repr, neg_emb], dim=-1)
        neg_rank     = self.ranking_tower(
            neg_combined).squeeze(-1)

        return (pos_rating, pos_completion,
                pos_rank, neg_rank)

    def recommend(self, history,
                  candidate_tokens,
                  rating_weight:     float = 0.4,
                  completion_weight: float = 0.3,
                  rank_weight:       float = 0.3):
        """
        Score candidates using all three tasks.
        Weighted combination for final ranking.
        """
        self.eval()
        with torch.no_grad():
            user_repr = self.encode_user(history)

            results = []
            for tok in candidate_tokens:
                item_emb = self.item_emb(
                    torch.LongTensor([[tok]]
                    ).to(history.device))
                item_emb = item_emb.squeeze(1)

                rating, completion, rank = \
                    self.predict_all_tasks(
                        user_repr, item_emb)

                # Normalised weighted score
                norm_rating = (
                    rating.item() - 0.5) / 4.5
                combined_score = (
                    rating_weight * norm_rating +
                    completion_weight *
                    completion.item() +
                    rank_weight * torch.sigmoid(
                        rank).item()
                )
                results.append((
                    tok,
                    combined_score,
                    rating.item(),
                    completion.item(),
                ))

            results.sort(
                key=lambda x: x[1],
                reverse=True)
            return results


mt_ranker = ExplicitMultiTaskRanker(
    vocab_size  = vocab_size,
    embed_dim   = 128,
    n_heads     = 4,
    n_layers    = 2,
    max_seq_len = 50,
).to(device)

n_params = sum(
    p.numel() for p in mt_ranker.parameters()
    if p.requires_grad)

print(f"✅ Multi-task Ranker instantiated")
print(f"   Parameters : {n_params:,}")
print(f"\nThree output towers:")
print(f"  Rating tower     → 0.5-5.0 ✅")
print(f"  Completion tower → 0.0-1.0 ✅")
print(f"  Ranking tower    → BPR score ✅")

✅ Multi-task Ranker instantiated
   Parameters : 1,679,491

Three output towers:
  Rating tower     → 0.5-5.0 ✅
  Completion tower → 0.0-1.0 ✅
  Ranking tower    → BPR score ✅


In [14]:
#Multi-task Loss
def multitask_loss(pos_rating,
                   pos_completion,
                   pos_rank,
                   neg_rank,
                   rating_true,
                   completion_true,
                   bpr_w:        float = 0.4,
                   rating_w:     float = 0.3,
                   completion_w: float = 0.3):
    """
    Explicit multi-task loss.
    BPR ranking    (0.4) → rank pos > neg
    Rating MSE     (0.3) → predict star rating
    Completion BCE (0.3) → predict watch %
    """
    # Flatten all tensors to 1D
    # handles both batch (256,) and single (1,)
    pos_rating       = pos_rating.view(-1)
    pos_completion   = pos_completion.view(-1)
    pos_rank         = pos_rank.view(-1)
    neg_rank         = neg_rank.view(-1)
    rating_true      = rating_true.view(-1)
    completion_true  = completion_true.view(-1)

    # BPR loss
    bpr = -F.logsigmoid(
        pos_rank - neg_rank).mean()

    # Rating MSE
    rating_loss = F.mse_loss(
        pos_rating, rating_true)

    # Completion BCE
    comp_loss = F.binary_cross_entropy(
        pos_completion.clamp(1e-6, 1-1e-6),
        completion_true.clamp(0.0, 1.0))

    total = (bpr_w        * bpr +
             rating_w     * rating_loss +
             completion_w * comp_loss)

    return total, {
        "bpr":        bpr.item(),
        "rating":     rating_loss.item(),
        "completion": comp_loss.item(),
        "total":      total.item(),
    }


print("✅ Multi-task loss updated")
print("""
Fix: view(-1) instead of squeeze()
  squeeze() removes ALL size-1 dimensions
  → scalar () when input is (1,)
  → shape mismatch crash

  view(-1) always gives 1D tensor
  → (256,) for batch
  → (1,)   for single event
  → shapes always match ✅
""")

✅ Multi-task loss updated

Fix: view(-1) instead of squeeze()
  squeeze() removes ALL size-1 dimensions
  → scalar () when input is (1,)
  → shape mismatch crash

  view(-1) always gives 1D tensor
  → (256,) for batch
  → (1,)   for single event
  → shapes always match ✅



In [15]:
# Train Multi-task Ranker
def train_mt(model, loader,
             optimizer, device):
    model.train()
    total_loss  = 0
    loss_parts  = defaultdict(float)
    n_batches   = 0

    for batch in loader:
        history    = batch['history'].to(device)
        targets    = batch['target'].to(device)
        negatives  = batch['negatives'].to(device)
        ratings    = batch['rating'].to(device)
        completions= batch['completion'].to(device)

        optimizer.zero_grad()

        pos_rating, pos_completion, \
        pos_rank, neg_rank = model(
            history,
            targets.squeeze(1),
            negatives[:, 0])

        loss, parts = multitask_loss(
            pos_rating, pos_completion,
            pos_rank, neg_rank,
            ratings, completions)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        for k, v in parts.items():
            loss_parts[k] += v
        n_batches += 1

    avg = {k: v/n_batches
           for k, v in loss_parts.items()}
    return avg


def eval_mt(model, loader, device):
    model.eval()
    total = 0
    n     = 0
    with torch.no_grad():
        for batch in loader:
            history    = batch['history']\
                .to(device)
            targets    = batch['target']\
                .to(device)
            negatives  = batch['negatives']\
                .to(device)
            ratings    = batch['rating']\
                .to(device)
            completions= batch['completion']\
                .to(device)

            pos_rating, pos_completion, \
            pos_rank, neg_rank = model(
                history,
                targets.squeeze(1),
                negatives[:, 0])

            loss, _ = multitask_loss(
                pos_rating, pos_completion,
                pos_rank, neg_rank,
                ratings, completions)

            total += loss.item()
            n     += 1
    return total / max(n, 1)


N_EPOCHS  = 20
optimizer = optim.Adam(
    mt_ranker.parameters(),
    lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler\
    .ReduceLROnPlateau(
        optimizer, patience=2,
        factor=0.5, verbose=False)

best_val   = float('inf')
patience_c = 0
tr_losses  = []
vl_losses  = []

mlflow.set_tracking_uri(
    settings.MLFLOW_TRACKING_URI)
mlflow.set_experiment(
    settings.MLFLOW_EXPERIMENT_NAME)

print(f"Training Multi-task Ranker")
print(f"{'Epoch':<8} {'Train':<12} "
      f"{'Val':<12} {'BPR':<10} "
      f"{'Rating':<10} {'Comp':<10} "
      f"{'Best':<5} Time")
print("─" * 75)

with mlflow.start_run(
        run_name="MultiTask_Ranker"):

    mlflow.log_params({
        "model":         "ExplicitMultiTask",
        "loss":          "BPR+MSE+BCE",
        "bpr_weight":    0.4,
        "rating_weight": 0.3,
        "comp_weight":   0.3,
        "towers":        3,
        "paper":         "Netflix 2025",
    })

    for epoch in range(1, N_EPOCHS + 1):
        start  = time.time()
        parts  = train_mt(
            mt_ranker, train_dl,
            optimizer, device)
        val_loss = eval_mt(
            mt_ranker, val_dl, device)
        scheduler.step(val_loss)
        elapsed = time.time() - start

        tr_losses.append(parts['total'])
        vl_losses.append(val_loss)

        mlflow.log_metrics({
            "train_loss":      parts['total'],
            "val_loss":        val_loss,
            "bpr_loss":        parts['bpr'],
            "rating_loss":     parts['rating'],
            "completion_loss": parts['completion'],
        }, step=epoch)

        is_best = val_loss < best_val
        if is_best:
            best_val   = val_loss
            patience_c = 0
            torch.save(
                mt_ranker.state_dict(),
                '../../models/checkpoints/'
                'multitask_ranker_best.pt')
        else:
            patience_c += 1

        print(
            f"{epoch:<8} "
            f"{parts['total']:<12.4f} "
            f"{val_loss:<12.4f} "
            f"{parts['bpr']:<10.4f} "
            f"{parts['rating']:<10.4f} "
            f"{parts['completion']:<10.4f} "
            f"{'✅' if is_best else '  ':<5} "
            f"{elapsed:.1f}s"
            + (f" p{patience_c}"
               if not is_best else ""))

        if patience_c >= 3:
            print(f"Early stop epoch {epoch}")
            break

    mlflow.log_metric(
        "best_val_loss", best_val)

print(f"\n✅ Multi-task training complete")
print(f"   Best val loss : {best_val:.4f}")

Training Multi-task Ranker
Epoch    Train        Val          BPR        Rating     Comp       Best  Time
───────────────────────────────────────────────────────────────────────────
1        0.3137       0.5309       0.2483     0.1683     0.5465     ✅     55.6s
2        0.3063       0.5569       0.2400     0.1573     0.5439           44.1s p1
3        0.3000       0.5379       0.2299     0.1512     0.5423           40.8s p2
4        0.2941       0.5317       0.2205     0.1453     0.5410           40.4s p3
Early stop epoch 4

✅ Multi-task training complete
   Best val loss : 0.5309


In [16]:
# Multi-task Recommendation Demo
mt_ranker.eval()

# Sample user
sample_uid = list(user_sequences.keys())[0]
seq        = user_sequences[sample_uid]

# Pad history
pad_l = 50 - len(seq)
hist  = torch.LongTensor(
    [[PAD_TOKEN]*pad_l + seq[-50:]]
).to(device)

# Get candidates
all_movie_tokens = list(movie2token.values())
candidates = list(np.random.choice(
    all_movie_tokens,
    size=min(100, len(all_movie_tokens)),
    replace=False))

# Rank with all three signals
ranked = mt_ranker.recommend(
    hist, candidates,
    rating_weight=0.4,
    completion_weight=0.3,
    rank_weight=0.3)

print(f"Multi-task Rankings for User {sample_uid}")
print("=" * 70)
print(f"{'Rank':<5} {'Title':<35} "
      f"{'Score':<10} "
      f"{'Rating':<10} {'Watch%':<8}")
print("─" * 70)

for i, (tok, score, rating, comp) in \
        enumerate(ranked[:10], 1):
    mid   = token2movie.get(int(tok))
    title = movies[
        movies['movieId'] == mid
    ]['title'].values if mid else []
    title = title[0][:33] \
        if len(title) > 0 else str(tok)
    print(f"{i:<5} {title:<35} "
          f"{score:<10.4f} "
          f"{rating:<10.2f} "
          f"{comp:<8.1%}")

print(f"""
Multi-task ranking insight:
  Score = 0.4×rating + 0.3×completion
          + 0.3×rank_score
  All three signals contribute to final rank
  A 5★ movie with 40% completion ranks LOWER
  than a 4★ movie with 90% completion
  → model learns to recommend films people finish
""")

Multi-task Rankings for User 1
Rank  Title                               Score      Rating     Watch%  
──────────────────────────────────────────────────────────────────────
1     Casablanca                          0.8591     4.25       75.3%   
2     I, Robot                            0.8361     4.01       74.9%   
3     Gosford Park                        0.8330     3.95       78.9%   
4     American Buffalo                    0.8323     4.09       72.6%   
5     Kolya                               0.8303     4.03       74.7%   
6     Hart's War                          0.8258     4.03       75.7%   
7     Something to Talk About             0.8207     3.91       73.9%   
8     The Shining                         0.8205     3.91       72.5%   
9     The Changeling                      0.8174     4.06       70.2%   
10    Black Dynamite                      0.8160     3.86       74.8%   

Multi-task ranking insight:
  Score = 0.4×rating + 0.3×completion
          + 0.3×rank_score
 

In [17]:
# Automated Retraining Trigger
print("AUTOMATED RETRAINING TRIGGER")
print("=" * 55)
print("Prefect flow: NDCG drop → retrain\n")

def compute_ndcg_at_k(model,
                       test_df,
                       user_sequences,
                       movie2token,
                       token2movie,
                       k=10,
                       n_users=100) -> float:
    """
    Compute NDCG@K on test set.
    Used to monitor model quality.
    """
    model.eval()
    ndcgs = []

    test_users = test_df['userId'].unique()
    eval_users = np.random.choice(
        test_users,
        size=min(n_users, len(test_users)),
        replace=False)

    user_relevant = {}
    for uid in eval_users:
        rel = set(test_df[
            (test_df['userId'] == uid) &
            (test_df['rating'] >= 4.0)
        ]['movieId'].values)
        if rel:
            user_relevant[uid] = rel

    all_tokens = list(movie2token.values())

    for uid in eval_users:
        relevant = user_relevant.get(uid)
        if not relevant:
            continue

        seq = user_sequences.get(uid, [])
        if not seq:
            continue

        pad_l = 50 - len(seq)
        hist  = torch.LongTensor(
            [[PAD_TOKEN]*pad_l + seq[-50:]]
        ).to(device)

        # Sample candidates
        cands = list(np.random.choice(
            all_tokens,
            size=min(200, len(all_tokens)),
            replace=False))

        ranked = model.recommend(hist, cands)
        rec_movies = [
            token2movie.get(int(tok))
            for tok, _, _, _ in ranked[:k]
            if token2movie.get(int(tok))
        ]

        # NDCG@K
        dcg  = sum(
            1.0 / np.log2(i + 2)
            for i, m in enumerate(rec_movies)
            if m in relevant
        )
        idcg = sum(
            1.0 / np.log2(i + 2)
            for i in range(
                min(len(relevant), k))
        )
        ndcgs.append(
            dcg / idcg if idcg > 0 else 0.0)

    return float(np.mean(ndcgs)) \
        if ndcgs else 0.0


class AutoRetrainTrigger:
    """
    Automated retraining trigger.
    Monitors NDCG@K — if it drops below
    threshold triggers retraining.

    Production version uses Prefect DAG.
    We implement the core logic here
    and show how Prefect would orchestrate it.
    """

    def __init__(self,
                 threshold:   float = 0.05,
                 patience:    int   = 3,
                 model_path:  str   = ''):
        self.threshold  = threshold
        self.patience   = patience
        self.model_path = model_path
        self.history    = []
        self.trigger_count = 0

    def check_and_trigger(self,
                           current_ndcg: float,
                           baseline_ndcg: float
                           ) -> dict:
        """
        Check if retraining should trigger.
        Returns action to take.
        """
        self.history.append(current_ndcg)
        drop = baseline_ndcg - current_ndcg
        drop_pct = drop / max(
            baseline_ndcg, 1e-10) * 100

        result = {
            "current_ndcg":  current_ndcg,
            "baseline_ndcg": baseline_ndcg,
            "drop":          round(drop, 4),
            "drop_pct":      round(drop_pct, 2),
            "trigger":       False,
            "action":        "monitor",
        }

        if drop_pct > self.threshold * 100:
            self.trigger_count += 1
            result["trigger"] = True
            result["action"]  = "retrain"
            result["message"] = (
                f"NDCG dropped {drop_pct:.1f}% "
                f"→ triggering retraining"
            )
        else:
            self.trigger_count = 0
            result["message"] = (
                f"NDCG healthy "
                f"(drop {drop_pct:.1f}% "
                f"< threshold "
                f"{self.threshold*100:.0f}%)")

        return result

    def prefect_flow_description(self):
        """Shows how Prefect would run this"""
        return """
PREFECT RETRAINING FLOW
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@flow(name="retraining_pipeline")
def retraining_flow():

    # 1. Monitor metrics
    ndcg = compute_ndcg_task()

    # 2. Check threshold
    if ndcg < THRESHOLD:

        # 3. Trigger retraining
        new_model = train_model_task()

        # 4. Evaluate new model
        new_ndcg = evaluate_task(new_model)

        # 5. Compare and deploy
        if new_ndcg > current_ndcg:
            deploy_model_task(new_model)
            mlflow.log_metric("ndcg", new_ndcg)
            notify_team("Model updated ✅")
        else:
            notify_team("Retrain failed ⚠️")

# Schedule: run every 6 hours
serve(retraining_flow,
      cron="0 */6 * * *")
"""


# Demo the trigger
print("Computing baseline NDCG...")
baseline_ndcg = compute_ndcg_at_k(
    mt_ranker, test_df,
    user_sequences,
    movie2token, token2movie,
    k=10, n_users=50)

print(f"Baseline NDCG@10: {baseline_ndcg:.4f}")

trigger = AutoRetrainTrigger(
    threshold=0.05,  # 5% drop triggers retrain
    patience=3)

# Simulate monitoring over time
print(f"\nSimulating model monitoring:")
print(f"{'Check':<8} {'NDCG':<10} "
      f"{'Drop%':<10} {'Action'}")
print("─" * 45)

simulated_ndcgs = [
    baseline_ndcg * 0.99,
    baseline_ndcg * 0.97,
    baseline_ndcg * 0.94,   # drops below 5%
    baseline_ndcg * 0.91,   # trigger
    baseline_ndcg * 1.02,   # after retrain
]

for i, ndcg in enumerate(
        simulated_ndcgs, 1):
    result = trigger.check_and_trigger(
        ndcg, baseline_ndcg)
    action = "🔴 RETRAIN" \
        if result['trigger'] \
        else "🟢 OK"
    print(f"{i:<8} {ndcg:<10.4f} "
          f"{result['drop_pct']:<10.1f} "
          f"{action}")

print(f"\n{trigger.prefect_flow_description()}")

AUTOMATED RETRAINING TRIGGER
Prefect flow: NDCG drop → retrain

Computing baseline NDCG...
Baseline NDCG@10: 0.0474

Simulating model monitoring:
Check    NDCG       Drop%      Action
─────────────────────────────────────────────
1        0.0469     1.0        🟢 OK
2        0.0459     3.0        🟢 OK
3        0.0445     6.0        🔴 RETRAIN
4        0.0431     9.0        🔴 RETRAIN
5        0.0483     -2.0       🟢 OK


PREFECT RETRAINING FLOW
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@flow(name="retraining_pipeline")
def retraining_flow():

    # 1. Monitor metrics
    ndcg = compute_ndcg_task()

    # 2. Check threshold
    if ndcg < THRESHOLD:

        # 3. Trigger retraining
        new_model = train_model_task()

        # 4. Evaluate new model
        new_ndcg = evaluate_task(new_model)

        # 5. Compare and deploy
        if new_ndcg > current_ndcg:
            deploy_model_task(new_model)
            mlflow.log_metric("ndcg", new_ndcg)
            notify_team("Model updated ✅

In [18]:
#Kafka Online Learning Loop
print("KAFKA ONLINE LEARNING LOOP")
print("=" * 55)
print("TikTok Monolith: interaction → update\n")

import json

class KafkaOnlineLearningSimulator:
    """
    Simulates the Kafka online learning loop.

    Production (TikTok Monolith):
    1. User interacts with recommendation
    2. Interaction event produced to Kafka
    3. Consumer reads event
    4. Lightweight embedding update applied
    5. Updated embedding served within seconds

    We simulate this with in-memory updates
    since we do not have a running Kafka in
    the notebook context.
    The docker-compose Kafka is used in
    Week 5 serving layer.
    """

    def __init__(self, model, device):
        self.model  = model
        self.device = device
        self.event_queue = []
        self.update_count = 0

    def produce_event(self,
                       user_id: int,
                       movie_id: int,
                       rating: float,
                       watch_pct: float):
        """Simulate producing event to Kafka"""
        event = {
            "user_id":    user_id,
            "movie_id":   movie_id,
            "rating":     rating,
            "watch_pct":  watch_pct,
            "timestamp":  datetime.now()\
                .isoformat(),
            "event_type": "interaction",
        }
        self.event_queue.append(event)
        return event

    def consume_and_update(self,
                            movie2token,
                            user_sequences):
        """
        Consume events and apply lightweight
        embedding updates.

        Full Monolith: updates shared embedding
        table in real time across all workers.
        Simplified: gradient step on new event.
        """
        if not self.event_queue:
            return 0

        events_processed = 0
        optimizer = optim.SGD(
            self.model.parameters(),
            lr=1e-4)  # small LR for online

        self.model.train()

        for event in self.event_queue:
            uid = event['user_id']
            mid = event['movie_id']

            if mid not in movie2token:
                continue

            seq = user_sequences.get(uid, [])
            if not seq:
                continue

            tok     = movie2token[mid]
            pad_l   = 50 - len(seq)
            hist    = torch.LongTensor(
                [[PAD_TOKEN]*pad_l + seq[-50:]]
            ).to(self.device)

            # Sample one negative
            all_toks = list(movie2token.values())
            neg_tok  = np.random.choice(all_toks)

            pos = torch.LongTensor([tok])\
                .to(self.device)
            neg = torch.LongTensor([neg_tok])\
                .to(self.device)
            r   = torch.FloatTensor(
                [event['rating']])\
                .to(self.device)
            c   = torch.FloatTensor(
                [event['watch_pct']])\
                .to(self.device)

            pr, pc, prk, nrk = self.model(
                hist, pos, neg)

            loss, _ = multitask_loss(
                pr, pc, prk, nrk, r, c)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            events_processed += 1
            self.update_count += 1

        self.event_queue.clear()
        self.model.eval()
        return events_processed


# Demo
simulator = KafkaOnlineLearningSimulator(
    mt_ranker, device)

# Simulate user interactions
sample_users = list(
    user_sequences.keys())[:5]

print("Producing interaction events to Kafka...")
for uid in sample_users:
    # Simulate a new interaction
    sample_mid = list(
        movie2token.keys())[
        np.random.randint(
            len(movie2token))]
    event = simulator.produce_event(
        user_id  = uid,
        movie_id = sample_mid,
        rating   = np.random.uniform(3.0, 5.0),
        watch_pct= np.random.uniform(0.5, 1.0),
    )
    print(f"  → Event: user={uid} "
          f"movie={sample_mid} "
          f"rating={event['rating']:.1f} "
          f"watch={event['watch_pct']:.0%}")

print(f"\nConsuming events + updating model...")
start = time.time()
n_updated = simulator.consume_and_update(
    movie2token, user_sequences)
elapsed = (time.time() - start) * 1000

print(f"  Updated: {n_updated} events")
print(f"  Time   : {elapsed:.0f}ms")
print(f"  Total updates: "
      f"{simulator.update_count}")
print(f"""
Production Kafka flow:
  Event produced      : <1ms
  Event consumed      : <10ms
  Embedding updated   : <50ms
  End-to-end latency  : <100ms

  TikTok Monolith achieves this at
  scale for 1B+ users
  Our simulation shows the same concept
  with full Kafka in Week 5 serving layer
""")

KAFKA ONLINE LEARNING LOOP
TikTok Monolith: interaction → update

Producing interaction events to Kafka...
  → Event: user=1 movie=147 rating=3.4 watch=92%
  → Event: user=2 movie=6957 rating=3.8 watch=68%
  → Event: user=3 movie=6772 rating=4.3 watch=56%
  → Event: user=4 movie=521 rating=4.8 watch=80%
  → Event: user=5 movie=64034 rating=3.2 watch=58%

Consuming events + updating model...
  Updated: 5 events
  Time   : 44ms
  Total updates: 5

Production Kafka flow:
  Event produced      : <1ms
  Event consumed      : <10ms
  Embedding updated   : <50ms
  End-to-end latency  : <100ms

  TikTok Monolith achieves this at
  scale for 1B+ users
  Our simulation shows the same concept
  with full Kafka in Week 5 serving layer



In [ ]:
#